In [24]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict,RandomizedSearchCV,GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline,make_pipeline
from xgboost import XGBRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, ConstantKernel as C,DotProduct, RationalQuadratic
from sklearn.neural_network import MLPRegressor
import shap
import warnings
warnings.filterwarnings("ignore")

In [25]:
# 初始准备工作
# 1.读取清洗好的数据
file_path = "Final_Features_Ultimate_all_deletion.csv" 
df = pd.read_csv(file_path)

In [26]:
# 2. 定义特征组 (严格对齐新版 CSV 列名)

# A. 力学特征组 (用于预测 Strength, Elongation)
# 包含：用量层 + 身份层 + 机理层
features_mech = [
    # --- 1. 用量层 (Dosage) ---
    'X1_Soft_Content',      # 软段含量 (原 X1_SoftSeg)
    'X2_DA_Content',        # DA单体含量
    'X3_Hard_Content',      # 硬段含量 (原 X3_HardSeg)
    'X4_R_Ratio',           # 异氰酸酯指数
    'X5_Add_Crosslink',     # 额外非DA交联剂含量 (原 X5_Crosslink)
    
    # --- 2. 身份层 (Identity) ---
    'DA_Linker_Type',       # 0=无, 1=糠胺, 2=糠醇 (替代原来的 DA_KA/KC)
    'Network_Topology',     # 0=主链, 1=侧链 (替代原来的 cross_class)
    'Add_Crosslinker_Flag', # 0=无, 1=有额外死交联 (替代 Has_Non_DA)
    'Polyol_Class',         # 0=聚醚, 1=聚酯 (替代 Polyol_Type)
    'Iso_Class',            # 0=脂肪族, 1=芳香族 (替代 Iso_Type 1-5分)
    
    # --- 3. 机理层 (Mechanism) ---
    'Soft_Mw',              # 软段分子量
    'Constraint_Factor',    # 冻结因子 (关键!)
    'Hard_Symmetry',        # 硬段对称性
    'Soft_Cryst',           # 软段结晶性
    'Synergy_Feature',      # 结晶协同因子
    'Extender_Class'        # 扩链剂刚性 (DAG=0, BQDO=1)
]

# B. 愈合特征组 (用于预测 Healing Efficiency)
# 包含：力学特征 + 工艺参数 (温度、时间)
features_heal = features_mech + ['healing_temperature', 'healing_time']

# 3. 二次清洗 (逻辑保持不变)
def get_clean_data(target_col, feature_cols, df):
    """
    清洗数据：仅保留目标值和特征值都不为空的行
    """
    # 检查列是否存在，防止报错
    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"以下特征在 CSV 中找不到，请检查列名拼写: {missing_cols}")

    # 剔除空值
    data = df.dropna(subset=[target_col] + feature_cols).copy()
    
    X = data[feature_cols].values
    y = data[target_col].values
    
    return X, y

# 4. 定义评估函数 (逻辑保持不变)
def evaluate_model(model, X, y, task_name="Task"):
    # 使用 LOOCV (留一法) 适合小样本 (<100)
    cv = LeaveOneOut() 
    
    # 预测
    # 注意：如果使用 GP 或 NeuralNet，建议在 Pipeline 里加 StandardScaler
    y_pred = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)
    
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    
    return r2, rmse, y_pred

In [ ]:
import pandas as pd
import numpy as np
import itertools

def generate_virtual_library_ultimate():
    print("正在根据 Final_Features_Ultimate.csv 结构生成虚拟配方库...")

    # === 1. 定义原材料基因库 (Design Space) ===
    
    # A. 软段定义: [名称, Polyol_Class(0醚/1酯), Soft_Mw, Soft_Cryst(0/1)]
    soft_segments = [
        # PTMG系列 (聚醚, 结晶)
        ('PTMG1000', 0, 1000, 1), 
        ('PTMG2000', 0, 2000, 1),
        ('PTMG3000', 0, 3000, 1), # 补充高分子量，用于验证愈合
        # PPG系列 (聚醚, 不结晶)
        ('PPG1000',  0, 1000, 0),
        ('PPG2000',  0, 2000, 0),
        # PCL系列 (聚酯, 结晶)
        ('PCL1000',  1, 1000, 1),
        ('PCL2000',  1, 2000, 1),
        ('PCL3000',  1, 3000, 1),
        # PBA系列 (聚酯, 结晶)
        ('PBA1000',  1, 1000, 1),
        ('PBA2000',  1, 2000, 1),
    ]

    # B. 硬段定义: [名称, Iso_Class(0脂/1芳), Hard_Symmetry(0否/1是)]
    hard_segments = [
        ('IPDI', 0, 0), # 脂肪族, 不对称 -> 软
        ('MDI',  1, 1), # 芳香族, 对称 -> 硬且强
        ('HDI',  0, 1), # 脂肪族, 对称 -> 软但规整
        ('TDI',  1, 0)  # 芳香族, 不对称 -> 硬但乱
    ]
    
    # C. 扩链剂定义: [名称, Extender_Class(0柔/1刚)]
    extenders = [
        ('BDO', 0),  # 普通扩链剂
        ('DAG', 0),  # 也是柔性 (注意之前的修正)
        ('BQDO', 1), # 刚性扩链剂 (用于提升模量)
    ]

    # D. 连续变量网格
    ranges = {
        'X1_Soft_Range': np.linspace(0.40, 0.85, 10), # 软段含量 40%~85%
        'X2_DA_Range':   np.linspace(0.05, 0.40, 8),  # DA含量 5%~40%
        'X4_Ratio':      [0.95, 0.98, 1.00, 1.02, 1.05],          # R值微调
        'X5_Add_Cross_Range': [0.0, 0.05]             # 额外死交联 (0或少量)
    }
    
    # E. 离散变量 (One-Hot / Categorical)
    # DA_Linker_Type: 1=KA(胺), 2=KC(醇)
    da_types = [1, 2] 
    
    # Network_Topology: 0=主链, 1=侧链
    topologies = [0, 1]

    virtual_data = []

    # === 2. 穷举循环 ===
    # 为了代码整洁，使用 itertools.product 简化嵌套循环
    combinations = itertools.product(
        soft_segments, 
        hard_segments, 
        extenders,
        da_types,
        topologies,
        ranges['X1_Soft_Range'],
        ranges['X2_DA_Range'],
        ranges['X4_Ratio'],
        ranges['X5_Add_Cross_Range']
    )

    for combo in combinations:
        (soft, hard, ext, da_type, topo, x1, x2, x4, x5) = combo
        
        s_name, s_class, s_mw, s_cryst = soft
        h_name, h_iso_class, h_sym = hard
        e_name, e_class = ext
        
        # --- A. 质量守恒与物理约束 ---
        # 硬段含量 = 1 - 软段 - DA - 额外死交联
        x3 = 1.0 - x1 - x2 - x5
        
        if x3 < 0.10: continue # 硬段太少，不成型
        if x3 > 0.60: continue # 硬段太多，太脆做不出来
        
        # --- B. 特征计算 (Feature Engineering) ---
        row = {}
        
        # 1. 用量层 (Dosage)
        row['X1_Soft_Content'] = x1
        row['X2_DA_Content'] = x2
        row['X3_Hard_Content'] = x3
        row['X4_R_Ratio'] = x4
        row['X5_Add_Crosslink'] = x5
        
        # 2. 身份层 (Identity)
        row['DA_Linker_Type'] = da_type
        row['Network_Topology'] = topo
        row['Add_Crosslinker_Flag'] = 1 if x5 > 0 else 0
        row['Polyol_Class'] = s_class
        row['Iso_Class'] = h_iso_class
        
        # 3. 机理层 (Mechanism) - 核心物理特征
        row['Soft_Mw'] = s_mw
        
        # 冻结因子 (Constraint Factor) = X3 / Soft_Mw
        # 注意单位问题，如果模型训练时用了归一化，这里也要注意量级。
        # 假设直接用数值：
        row['Constraint_Factor'] = x3 / s_mw if s_mw > 0 else 0
        
        row['Hard_Symmetry'] = h_sym
        row['Soft_Cryst'] = s_cryst
        
        # 协同因子 (Synergy)
        row['Synergy_Feature'] = x1 * s_cryst * h_sym
        
        # 扩链剂刚性
        row['Extender_Class'] = e_class

        # --- C. 辅助信息 (用于人类阅读/筛选) ---
        # 生成一个可读的配方名称，方便后续查找
        # 格式: Soft_Hard_DA_Ext_X1_X2
        da_str = "KA" if da_type == 1 else "KC"
        topo_str = "Main" if topo == 0 else "Side"
        row['Recipe_Name'] = f"{s_name}_{h_name}_{da_str}({topo_str})_{e_name}_X1={x1:.2f}_X2={x2:.2f}"
        
        # --- D. 工艺参数 (默认值) ---
        # 预测时模型需要这些列，虽然对结果影响可能不大(如果SHAP低)
        row['poly_tem'] = 80
        row['strain_rate'] = 50
        row['healing_temperature'] = 60
        row['healing_time'] = 24

        virtual_data.append(row)

    # === 3. 转换为 DataFrame ===
    df_virtual = pd.DataFrame(virtual_data)
    
    # 再次检查列名是否与 features_mech / features_heal 对齐
    # 这里不做剔除，保留所有列，预测时再选
    return df_virtual

# 执行生成
df_virtual_lib = generate_virtual_library_ultimate()
print(f"虚拟配方库生成完毕，共 {len(df_virtual_lib)} 条数据")
print("前5行预览:")
print(df_virtual_lib[['Recipe_Name']].head())

正在根据 Final_Features_Ultimate.csv 结构生成虚拟配方库...
虚拟配方库生成完毕，共 206400 条数据
前5行预览:
                                  Recipe_Name
0  PTMG1000_IPDI_KA(Main)_BDO_X1=0.40_X2=0.05
1  PTMG1000_IPDI_KA(Main)_BDO_X1=0.40_X2=0.05
2  PTMG1000_IPDI_KA(Main)_BDO_X1=0.40_X2=0.05
3  PTMG1000_IPDI_KA(Main)_BDO_X1=0.40_X2=0.05
4  PTMG1000_IPDI_KA(Main)_BDO_X1=0.40_X2=0.05


In [39]:
# 最优模型选择
# 拉伸强度 (Strength) - SVR
target_col = 'tensile_strength'  
features = features_mech
X, y = get_clean_data(target_col, features,df)

param_grid = {'svr__C': [1, 10, 50, 100, 500, 1000], 'svr__gamma': ['scale', 0.001, 0.01, 0.1, 0.5, 1.0],'svr__epsilon': [0.01, 0.1, 0.5]}

pipe = make_pipeline(StandardScaler(), SVR(kernel='rbf'))

grid_search = GridSearchCV(pipe, param_grid, cv=LeaveOneOut(), scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

grid_search.fit(X, y)

# 获取最佳模型
best_svr_tensile = grid_search.best_estimator_
best_params = grid_search.best_params_

print("\n[最佳参数组合]")
print(best_params)

#训练
data_str = df.dropna(subset=['tensile_strength'] + features_mech).copy()

X_str = data_str[features_mech]
y_str = data_str['tensile_strength']

best_svr_tensile.fit(X_str, y_str)
print(f"强度模型训练完成。有效样本数：{len(X_str)}")

r2, rmse, _ = evaluate_model(best_svr_tensile, X_str, y_str, "Strength")
print(f"Strength | R2={r2:.3f}, RMSE={rmse:.2f}")



Fitting 84 folds for each of 108 candidates, totalling 9072 fits

[最佳参数组合]
{'svr__C': 100, 'svr__epsilon': 0.01, 'svr__gamma': 0.01}
强度模型训练完成。有效样本数：84
Strength | R2=0.765, RMSE=6.52


In [40]:
# 2. 断裂伸长率 (Elongation) - SVR
target_col = 'elongation'  
features = features_mech
X, y = get_clean_data(target_col, features, df)

param_grid = {'svr__C': [1, 10, 50, 100, 500, 1000], 'svr__gamma': ['scale', 0.001, 0.01, 0.1, 0.5, 1.0],'svr__epsilon': [0.01, 0.1, 0.5]
}

pipe = make_pipeline(StandardScaler(), SVR(kernel='rbf'))

grid_search = GridSearchCV(pipe, param_grid, cv=LeaveOneOut(),scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

grid_search.fit(X, y)

# 获取最佳模型
best_svr_elongation = grid_search.best_estimator_
best_params = grid_search.best_params_

print("\n[最佳参数组合]")
print(best_params)

# 训练
data_elo = df.dropna(subset=['elongation'] + features_mech).copy()

X_elo = data_elo[features_mech]
y_elo = data_elo['elongation']

best_svr_elongation.fit(X_elo, y_elo)
print(f"伸长率模型训练完成。有效样本数: {len(X_elo)}")

r2, rmse, _ = evaluate_model(best_svr_elongation, X_elo, y_elo, "elongation")
print(f"elongation | R2={r2:.3f}, RMSE={rmse:.2f}")

Fitting 84 folds for each of 108 candidates, totalling 9072 fits

[最佳参数组合]
{'svr__C': 1000, 'svr__epsilon': 0.5, 'svr__gamma': 0.5}
伸长率模型训练完成。有效样本数: 84
elongation | R2=0.615, RMSE=167.29


In [41]:

# 3. 自愈合 (Healing) - SVR
target_col = 'healing_eff'
features = features_heal
X, y = get_clean_data(target_col, features, df)

param_grid = {'svr__C': [1, 10, 50, 100, 500, 1000], 'svr__gamma': ['scale', 0.001, 0.01, 0.1, 0.5, 1.0],'svr__epsilon': [0.01, 0.1, 0.5]}

pipe = make_pipeline(StandardScaler(), SVR(kernel='rbf'))

grid_search = GridSearchCV(pipe, param_grid, cv=LeaveOneOut(), scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

grid_search.fit(X, y)

# 获取最佳模型
best_svr_healing = grid_search.best_estimator_
best_params = grid_search.best_params_

print("\n[最佳参数组合]")
print(best_params)

#3.自愈合率
data_heal = df.dropna(subset=['healing_eff'] + features_heal).copy()

X_heal = data_heal[features_heal]
y_heal = data_heal['healing_eff']

best_svr_healing.fit(X_heal, y_heal)
print(f"自愈合模型训练完成。有效样本数: {len(X_heal)}")

r2, rmse, _ = evaluate_model(best_svr_healing, X_heal, y_heal, "healing_eff")
print(f"healing_eff | R2={r2:.3f}, RMSE={rmse:.2f}")

Fitting 37 folds for each of 108 candidates, totalling 3996 fits

[最佳参数组合]
{'svr__C': 100, 'svr__epsilon': 0.5, 'svr__gamma': 0.01}
自愈合模型训练完成。有效样本数: 37
healing_eff | R2=0.798, RMSE=5.43


In [43]:
# === 3. 预测与筛选 (适配全能型特征仓库) ===

# A. 定义预测所需的特征列表 (必须与模型训练时的 features_mech 完全一致！)
# 请检查这 16 个特征是否与你训练 SVR 时的列表顺序一致
features_mech_pred = [
    'X1_Soft_Content', 'X2_DA_Content', 'X3_Hard_Content', 'X4_R_Ratio', 'X5_Add_Crosslink',
    'DA_Linker_Type', 'Network_Topology', 'Add_Crosslinker_Flag', 'Polyol_Class', 'Iso_Class',
    'Soft_Mw', 'Constraint_Factor', 'Hard_Symmetry', 'Soft_Cryst', 'Synergy_Feature', 'Extender_Class'
]

# B. 定义自愈合模型的特征列表 (力学特征 + 工艺参数)
features_heal_pred = features_mech_pred + ['healing_temperature', 'healing_time']

# C. 准备预测数据
try:
    # 1. 确保虚拟库里有默认的愈合工艺参数 (如果生成时没加)
    if 'healing_temperature' not in df_virtual_lib.columns:
        df_virtual_lib['healing_temperature'] = 60.0
    if 'healing_time' not in df_virtual_lib.columns:
        df_virtual_lib['healing_time'] = 24.0
        
    # 2. 提取特征矩阵 (自动对齐列名)
    X_input_mech = df_virtual_lib[features_mech_pred]
    X_input_heal = df_virtual_lib[features_heal_pred]
    
except KeyError as e:
    print(f"❌ 列名缺失错误: {e}")
    print("请检查 df_virtual_lib 的列名是否与 features_mech_pred 一致。")
    # 打印差异，方便调试
    missing = [c for c in features_mech_pred if c not in df_virtual_lib.columns]
    print(f"缺失的列: {missing}")
    raise

# D. 执行预测
print(f"正在对 {len(df_virtual_lib)} 个虚拟配方进行全量预测...")

# 预测强度 (SVR)
df_virtual_lib['Pred_Strength'] = best_svr_tensile.predict(X_input_mech)

# 预测伸长率 (SVR)
df_virtual_lib['Pred_Elongation'] = best_svr_elongation.predict(X_input_mech)

# 预测自愈合 (SVR)
df_virtual_lib['Pred_Healing'] = best_svr_healing.predict(X_input_heal)

print("预测完成！")

# === 统计概览 ===
print("\n=== 预测值统计概览 ===")
print(df_virtual_lib[['Pred_Strength', 'Pred_Healing', 'Pred_Elongation']].describe().round(2))

# --- 极值检查 (Sanity Check) ---
print("\n=== 最硬配方 Top 3 (检查强度上限) ===")
cols_debug = ['Recipe_Name', 'Pred_Strength', 'Pred_Healing', 'Pred_Elongation']
print(df_virtual_lib.sort_values(by='Pred_Strength', ascending=False)[cols_debug].head(3).to_string(index=False))

print("\n=== 最易愈合配方 Top 3 (检查愈合上限) ===")
print(df_virtual_lib.sort_values(by='Pred_Healing', ascending=False)[cols_debug].head(3).to_string(index=False))


# === 4. 筛选 (Target A & B) ===

# 策略 A: 刚性锚点 (Rigid Anchor)
# 逻辑：强度要高 (>30)，愈合可以妥协但最好有点 (>10)，伸长率不强求
# 注意：如果 SVR 预测的强度普遍偏低，可以适当下调阈值到 25
target_A = df_virtual_lib[
    (df_virtual_lib['Pred_Strength'] > 30) &      
    (df_virtual_lib['Pred_Strength'] < 35) &  
    (df_virtual_lib['Pred_Healing'] < 65) &  # 验证“冻结效应”，愈合应该被压制     
    (df_virtual_lib['Pred_Elongation'] > 50) # 哪怕是硬材料，也希望能成膜
].sort_values(by='Pred_Strength', ascending=False)

# 策略 B: 动态流体 (Dynamic Fluid)
# 逻辑：愈合要极高 (>80)，强度适中 (5-20)，伸长率越高越好
target_B = df_virtual_lib[
    (df_virtual_lib['Pred_Healing'] > 80) &
    (df_virtual_lib['Pred_Healing'] < 95) &       
    (df_virtual_lib['Pred_Strength'] > 5) &       
    (df_virtual_lib['Pred_Strength'] < 20)        
].sort_values(by='Pred_Healing', ascending=False)

# === 5. 展示与导出 ===
# 更新展示的列名 (对应新特征)
cols_to_show = ['Recipe_Name', 'X1_Soft_Content', 'X3_Hard_Content', 'Soft_Mw', 'Constraint_Factor', 
                'Pred_Strength', 'Pred_Healing', 'Pred_Elongation']

print("\n====== Target A: 推荐的【刚性锚点】配方 (Top 20) ======")
if len(target_A) > 0:
    print(target_A[cols_to_show].head(20).to_string(index=False))
else:
    print("未找到满足 Target A 严苛条件的配方，建议放宽强度阈值。")
print("\n====== Target B: 推荐的【动态流体】配方 (Top 20) ======")
if len(target_B) > 0:
    print(target_B[cols_to_show].head(20).to_string(index=False))
else:
    print("未找到满足 Target B 严苛条件的配方，建议放宽愈合阈值。")

# 导出结果
target_A.head(25).to_csv('Target_A_Recipes_New.csv', index=False)
target_B.head(25).to_csv('Target_B_Recipes_New.csv', index=False)
print("\n配方已保存至 Target_A_Recipes_New.csv 和 Target_B_Recipes_New.csv")

正在对 247680 个虚拟配方进行全量预测...
预测完成！

=== 预测值统计概览 ===
       Pred_Strength  Pred_Healing  Pred_Elongation
count      247680.00     247680.00        247680.00
mean           23.39         77.35           521.81
std            11.02          9.64            11.49
min           -19.25         43.44           364.63
25%            19.19         70.47           521.37
50%            20.50         77.58           521.37
75%            26.30         84.61           521.37
max            69.63        101.81           928.06

=== 最硬配方 Top 3 (检查强度上限) ===
                              Recipe_Name  Pred_Strength  Pred_Healing  Pred_Elongation
PTMG4000_MDI_KC(Main)_BDO_X1=0.40_X2=0.05      69.626848     65.219594       521.371481
PTMG4000_MDI_KC(Main)_DAG_X1=0.40_X2=0.05      69.626848     65.219594       521.371481
PTMG4000_MDI_KC(Main)_DAG_X1=0.40_X2=0.10      69.604079     65.345217       521.372523

=== 最易愈合配方 Top 3 (检查愈合上限) ===
                              Recipe_Name  Pred_Strength  Pred_Healing 

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def identify_pareto_3d(df, cols=['Pred_Strength', 'Pred_Elongation', 'Pred_Healing']):
    """
    计算 3D 帕累托前沿点 (非支配解)
    如果一个点在三个维度上都不被其他点完全压制，它就是帕累托最优的。
    """
    population = df[cols].values
    is_pareto = np.ones(population.shape[0], dtype=bool)
    
    # 简单的两重循环筛选 (数据量大时稍慢，但逻辑清晰)
    # 对于几千个点没问题
    for i, c in enumerate(population):
        if is_pareto[i]:
            # 如果存在任何一个点 j，在三项指标上都 >= 点 i，且至少有一项 > 点 i
            # 那么点 i 就不是帕累托最优
            # 注意：这里假设三个指标都是越大越好
            is_dominated = np.any(np.all(population >= c, axis=1) & np.any(population > c, axis=1))
            if is_dominated:
                is_pareto[i] = False
                
    return df[is_pareto]



def plot_3d_material_space_plotly(df_virtual, target_A, target_B):
    # 1. Pareto Front
    pareto_df = identify_pareto_3d(df_virtual)
    print(f"Pareto 最优解数量: {len(pareto_df)}")

    # 2. 背景虚拟库
    background = go.Scatter3d(
        x=df_virtual['Pred_Strength'],
        y=df_virtual['Pred_Elongation'],
        z=df_virtual['Pred_Healing'],
        mode='markers',
        marker=dict(
            size=3,
            color='lightgray',
            opacity=0.15
        ),
        name='Virtual Library',
        hoverinfo='skip'
    )

    # 3. Pareto Surface
    pareto = go.Scatter3d(
        x=pareto_df['Pred_Strength'],
        y=pareto_df['Pred_Elongation'],
        z=pareto_df['Pred_Healing'],
        mode='markers',
        marker=dict(
            size=5,
            color=pareto_df['Pred_Healing'],
            colorscale='Viridis',
            opacity=0.9,
            colorbar=dict(title='Healing Efficiency')
        ),
        name='Pareto Front',
        hovertemplate=
        'Strength: %{x:.2f}<br>' +
        'Elongation: %{y:.2f}<br>' +
        'Healing: %{z:.2f}<extra></extra>'
    )

    # 4. Target A
    target_a = go.Scatter3d(
    x=[target_A['Pred_Strength']],
    y=[target_A['Pred_Elongation']],
    z=[target_A['Pred_Healing']],
    mode='markers',
    marker=dict(
        size=12,
        color='red',
        symbol='diamond',
        line=dict(color='black', width=2)
    ),
    name='Target A (Rigid)'
)

    # 5. Target B
    target_b = go.Scatter3d(
    x=[target_B['Pred_Strength']],
    y=[target_B['Pred_Elongation']],
    z=[target_B['Pred_Healing']],
    mode='markers',
    marker=dict(
        size=12,
        color='blue',
        symbol='diamond',
        line=dict(color='black', width=2)
    ),
    name='Target B (Dynamic)'
)

    # 6. Layout
    layout = go.Layout(
        title='3D Pareto Frontier: Strength vs Elongation vs Healing',
        scene=dict(
            xaxis=dict(title='Strength (MPa)'),
            yaxis=dict(title='Elongation (%)'),
            zaxis=dict(title='Healing Efficiency (%)')
        ),
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig = go.Figure(data=[background, pareto, target_a, target_b], layout=layout)
    fig.show()

plot_3d_material_space_plotly(df_virtual, target_A, target_B)
